# Repartition vs Coalesce

Both `repartition` and `coalesce` change the number of partitions in a DataFrame, but they do it in fundamentally different ways:

* `repartition(n[, col])` introduces a **shuffle**. The previous stage runs with its existing parallelism and the new stage with `n` evenly-balanced partitions. Can both increase and decrease.
* `coalesce(n)` is a **narrow dependency** — it merges existing partitions without a shuffle. It cannot increase the partition count, output partitions can be skewed if the input was, and — most importantly — it limits the parallelism of the *upstream* stage as well, because there is no shuffle boundary to isolate the two.

We also look at `spark.sql.shuffle.partitions`, which controls the partition count produced by any shuffle (groupBy, join, ...).

Tasks:
1. Observe the baseline partition count after reading the data.
2. Compare the physical plans of `coalesce(n)` and `repartition(n)`.
3. See the upstream-starvation trap of `coalesce`.
4. Use `repartition(n, col)` to align downstream operations on the same key.
5. Control the number of output files when writing.
6. See how `spark.sql.shuffle.partitions` interacts with the above.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, count, lower

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Repartition vs Coalesce')
    .getOrCreate()
)

In [ ]:
print(spark.version)

#### Notes about the session:

* we turn AQE off so the static plan we read with `.explain()` matches what actually runs — AQE coalesces shuffle partitions at runtime, which would obscure the comparison
* we keep the default `spark.sql.shuffle.partitions = 200` for now

In [ ]:
spark.conf.set('spark.sql.adaptive.enabled', False)

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

questions_input_path = os.path.join(project_path, 'data/questions-json')

questions_output_path = os.path.join(project_path, 'output/questions-files')

In [ ]:
questionsDF = spark.read.json(questions_input_path)

### Task 1: Baseline partition count

Before changing anything, see how many partitions the source DataFrame has. Spark decides this based on the input files and the value of `spark.sql.files.maxPartitionBytes`.

Hint:
* call `questionsDF.rdd.getNumPartitions()`

In [ ]:
print('input partitions:', questionsDF.rdd.getNumPartitions())

### Task 2: `coalesce(n)` vs `repartition(n)` — compare the plans

Both calls below request 4 output partitions. Look at the physical plans and notice that `coalesce(4)` produces a `Coalesce 4` node (no `Exchange`), while `repartition(4)` produces a `Exchange RoundRobinPartitioning(4)` — that `Exchange` is the shuffle.

Hint:
* `df.coalesce(4).explain()` — no Exchange in the plan
* `df.repartition(4).explain()` — Exchange present
* check the partition count of the resulting DataFrame with `.rdd.getNumPartitions()`

In [ ]:
coalescedDF = questionsDF.coalesce(4)

coalescedDF.explain()

print('partitions after coalesce(4):', coalescedDF.rdd.getNumPartitions())

In [ ]:
repartitionedDF = questionsDF.repartition(4)

repartitionedDF.explain()

print('partitions after repartition(4):', repartitionedDF.rdd.getNumPartitions())

### Task 3: The upstream-starvation trap of `coalesce`

Because `coalesce` is a narrow dependency, the executor running each coalesced output partition has to run all of the upstream work that feeds into it. So `coalesce(4)` at the end of a pipeline does not just collapse the *write* into 4 tasks — it collapses the *entire upstream computation* into 4 tasks as well.

Run the two cells below — both write the same filtered/projected questions data with 4 output partitions. Then go to the Spark UI's Stages tab and compare the number of tasks in the read stage:
* with `coalesce(4)` — the read + filter + project stage runs with **4 tasks**
* with `repartition(4)` — the read + filter + project stage runs with the full input parallelism, and only the final write stage has 4 tasks

Hint:
* if the upstream work is expensive (CPU-bound transformations, UDFs, reading many files), `coalesce` can be catastrophically slow
* rule of thumb: use `coalesce` only when the upstream is already cheap or already has at most `n` partitions

In [ ]:
(
    questionsDF
    .filter(col('score') > 0)
    .withColumn('body_len', length('body'))
    .withColumn('title_lc', lower(col('title')))
    .coalesce(4)
    .write
    .mode('overwrite')
    .format('noop')
    .save()
)

In [ ]:
(
    questionsDF
    .filter(col('score') > 0)
    .withColumn('body_len', length('body'))
    .withColumn('title_lc', lower(col('title')))
    .repartition(4)
    .write
    .mode('overwrite')
    .format('noop')
    .save()
)

### Task 4: `repartition(n, col)` to align downstream operations

Adding a column argument hash-partitions the data by that column. If the next operation is a `groupBy`/`join` on the same column, Spark can skip its own shuffle — the data is already on the right executor. This is the same trick used in `Optimize-I.ipynb`, included here for completeness.

Hint:
* compare the plans with and without `repartition(8, 'user_id')` before a `groupBy('user_id')`
* with the repartition, the groupBy does not introduce a second `Exchange`

In [ ]:
no_repartition = (
    questionsDF
    .groupBy('user_id')
    .agg(count('*').alias('q_count'))
)

no_repartition.explain()

In [ ]:
with_repartition = (
    questionsDF
    .repartition(8, 'user_id')
    .groupBy('user_id')
    .agg(count('*').alias('q_count'))
)

with_repartition.explain()

### Task 5: Controlling the number of output files when writing

The number of files Spark writes is equal to the number of partitions of the final DataFrame (assuming no `partitionBy` / `bucketBy`). To produce a specific file count you place a `repartition(n)` or `coalesce(n)` right before the write. The same upstream-starvation caveat from Task 3 applies — prefer `repartition` if the upstream pipeline is non-trivial.

Hint:
* write once with `repartition(4)` and once with `coalesce(4)` and compare the number of `part-*.parquet` files produced (and their sizes)
* you can list the output folder with `os.listdir(...)` to count files

In [ ]:
(
    questionsDF
    .repartition(4)
    .write
    .mode('overwrite')
    .parquet(questions_output_path)
)

files = sorted(f for f in os.listdir(questions_output_path) if f.startswith('part-'))

print('files written with repartition(4):', len(files))
for f in files:
    size_kb = os.path.getsize(os.path.join(questions_output_path, f)) // 1024
    print(f'  {f}  —  {size_kb} KB')

In [ ]:
(
    questionsDF
    .coalesce(4)
    .write
    .mode('overwrite')
    .parquet(questions_output_path)
)

files = sorted(f for f in os.listdir(questions_output_path) if f.startswith('part-'))

print('files written with coalesce(4):', len(files))
for f in files:
    size_kb = os.path.getsize(os.path.join(questions_output_path, f)) // 1024
    print(f'  {f}  —  {size_kb} KB')

### Task 6: `spark.sql.shuffle.partitions`

Every wide transformation (groupBy, join, window, ...) produces a new partitioning whose width is `spark.sql.shuffle.partitions` (default 200). With AQE on, Spark dynamically coalesces these down when the actual data is smaller; with AQE off (as in this notebook), 200 is exactly what you get — and for small inputs that means hundreds of empty tasks.

Run a groupBy with the default and then with the value reduced to 10, and compare:

Hint:
* `spark.conf.set('spark.sql.shuffle.partitions', 10)`
* after the groupBy, check `.rdd.getNumPartitions()` on the result
* go to the Stages tab to see how many tasks run in the aggregation stage

In [ ]:
spark.conf.set('spark.sql.shuffle.partitions', 200)

result_default = questionsDF.groupBy('user_id').agg(count('*').alias('q_count'))

result_default.write.mode('overwrite').format('noop').save()

print('partitions after groupBy with shuffle.partitions=200:', result_default.rdd.getNumPartitions())

In [ ]:
spark.conf.set('spark.sql.shuffle.partitions', 10)

result_small = questionsDF.groupBy('user_id').agg(count('*').alias('q_count'))

result_small.write.mode('overwrite').format('noop').save()

print('partitions after groupBy with shuffle.partitions=10:', result_small.rdd.getNumPartitions())

In [ ]:
spark.stop()